<a href="https://colab.research.google.com/github/MuhammadRayan07/Weather-Analysis/blob/main/notebooks/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weather Data Analysis and Prediction
## Notebook 1: Data Collection

**Purpose:** Collect daily historical weather data for 20 cities (2016-2025) from the Open-Meteo Historical Weather API.
**Data source:** Open-Meteo (https://open-meteo.com), reanalysis-based data.


In [9]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path("/content/drive/MyDrive/weather-analysis")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Folders ready:", RAW.exists(), PROCESSED.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folders ready: True True


In [10]:
import pandas as pd

cities = pd.DataFrame([
    ("Lahore","Pakistan",31.5497,74.3436),("Karachi","Pakistan",24.8607,67.0011),
    ("Dubai","UAE",25.2048,55.2708),("Delhi","India",28.6139,77.2090),
    ("Tokyo","Japan",35.6762,139.6503),("Singapore","Singapore",1.3521,103.8198),
    ("Beijing","China",39.9042,116.4074),("Sydney","Australia",-33.8688,151.2093),
    ("Cairo","Egypt",30.0444,31.2357),("Nairobi","Kenya",-1.2921,36.8219),
    ("Lagos","Nigeria",6.5244,3.3792),("Cape Town","South Africa",-33.9249,18.4241),
    ("London","UK",51.5074,-0.1278),("Paris","France",48.8566,2.3522),
    ("Moscow","Russia",55.7558,37.6173),("Reykjavik","Iceland",64.1466,-21.9426),
    ("New York","USA",40.7128,-74.0060),("Toronto","Canada",43.6532,-79.3832),
    ("Mexico City","Mexico",19.4326,-99.1332),("Sao Paulo","Brazil",-23.5505,-46.6333),
], columns=["city","country","latitude","longitude"])

cities.to_csv(PROCESSED / "city_reference.csv", index=False)
cities

,city,country,latitude,longitude
0,Lahore,Pakistan,31.5497,74.3436
1,Karachi,Pakistan,24.8607,67.0011
2,Dubai,UAE,25.2048,55.2708
3,Delhi,India,28.6139,77.2090
4,Tokyo,Japan,35.6762,139.6503
5,Singapore,Singapore,1.3521,103.8198
6,Beijing,China,39.9042,116.4074
7,Sydney,Australia,-33.8688,151.2093
8,Cairo,Egypt,30.0444,31.2357
9,Nairobi,Kenya,-1.2921,36.8219


Choosing cities with different weather conditions (a mix of tropical, desert, temperate, and polar climates across six continents)

In [11]:
import requests

URL = "https://archive-api.open-meteo.com/v1/archive"
DAILY = ["weather_code","temperature_2m_max","temperature_2m_min","temperature_2m_mean",
         "precipitation_sum","rain_sum","precipitation_hours",
         "wind_speed_10m_max","wind_gusts_10m_max","wind_direction_10m_dominant",
         "shortwave_radiation_sum","sunshine_duration","daylight_duration"]

test = cities.iloc[0]
params = {"latitude": test.latitude, "longitude": test.longitude,
          "start_date": "2016-01-01", "end_date": "2025-12-31",
          "daily": ",".join(DAILY), "timezone": "auto"}

r = requests.get(URL , params=params ,timeout=60)
print("Status:", r.status_code)
data = r.json()
if r.status_code != 200:
    print(data)
else:
    print(data.keys())
    print(data["daily_units"])
    print(len(data["daily"]["time"]), "days")

Status: 200
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
{'time': 'iso8601', 'weather_code': 'wmo code', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'temperature_2m_mean': '°C', 'precipitation_sum': 'mm', 'rain_sum': 'mm', 'precipitation_hours': 'h', 'wind_speed_10m_max': 'km/h', 'wind_gusts_10m_max': 'km/h', 'wind_direction_10m_dominant': '°', 'shortwave_radiation_sum': 'MJ/m²', 'sunshine_duration': 's', 'daylight_duration': 's'}
3653 days


In [12]:
import time
extra_candidates = ["relative_humidity_2m_mean", "dew_point_2m_mean",
                    "cloud_cover_mean", "pressure_msl_mean", "surface_pressure_mean"]

accepted = []
for var in extra_candidates:
    p = {"latitude": 31.5497, "longitude": 74.3436,
         "start_date": "2016-01-01", "end_date": "2016-01-10",
         "daily": var, "timezone": "auto"}
    resp = requests.get(URL, params=p, timeout=60)
    ok = resp.status_code == 200
    print(f"{var:30s} -> {'OK' if ok else 'REJECTED: ' + str(resp.json().get('reason'))}")
    if ok:
        accepted.append(var)
    time.sleep(1)

print("\nAccepted extras:", accepted)

relative_humidity_2m_mean      -> REJECTED: Daily API request limit exceeded. Please try again tomorrow.
dew_point_2m_mean              -> REJECTED: Daily API request limit exceeded. Please try again tomorrow.
cloud_cover_mean               -> REJECTED: Daily API request limit exceeded. Please try again tomorrow.
pressure_msl_mean              -> REJECTED: Daily API request limit exceeded. Please try again tomorrow.
surface_pressure_mean          -> REJECTED: Daily API request limit exceeded. Please try again tomorrow.

Accepted extras: []


In [13]:
import json

DAILY_ALL = DAILY + accepted

def fetch_city(row, max_retries=5):
    params = {"latitude": row.latitude, "longitude": row.longitude,
              "start_date": "2016-01-01", "end_date": "2025-12-31",
              "daily": ",".join(DAILY_ALL), "timezone": "auto"}
    for attempt in range(max_retries):
        r = requests.get(URL, params=params, timeout=120)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            print(f"  Rate limited, waiting 65s (attempt {attempt+1})")
            time.sleep(65)
        else:
            print("  Error:", r.status_code, r.text[:200])
            break
    return None

log = []
for row in cities.itertuples():
    raw_file = RAW / f"{row.city.replace(' ', '_')}.json"
    if raw_file.exists():
        data = json.loads(raw_file.read_text())
        print(f"{row.city}: loaded from saved file")
    else:
        print(f"{row.city}: requesting...")
        data = fetch_city(row)
        if data is not None:
            raw_file.write_text(json.dumps(data))
            time.sleep(3)
    log.append({"city": row.city,
                "success": data is not None,
                "days_returned": len(data["daily"]["time"]) if data else 0})

collection_log = pd.DataFrame(log)
collection_log["expected_days"] = 3653
collection_log["missing_days"] = collection_log["expected_days"] - collection_log["days_returned"]
collection_log

Lahore: loaded from saved file
Karachi: loaded from saved file
Dubai: loaded from saved file
Delhi: loaded from saved file
Tokyo: loaded from saved file
Singapore: loaded from saved file
Beijing: loaded from saved file
Sydney: loaded from saved file
Cairo: loaded from saved file
Nairobi: loaded from saved file
Lagos: loaded from saved file
Cape Town: loaded from saved file
London: loaded from saved file
Paris: loaded from saved file
Moscow: loaded from saved file
Reykjavik: loaded from saved file
New York: loaded from saved file
Toronto: loaded from saved file
Mexico City: loaded from saved file
Sao Paulo: loaded from saved file


,city,success,days_returned,expected_days,missing_days
0,Lahore,True,3653,3653,0
1,Karachi,True,3653,3653,0
2,Dubai,True,3653,3653,0
3,Delhi,True,3653,3653,0
4,Tokyo,True,3653,3653,0
5,Singapore,True,3653,3653,0
6,Beijing,True,3653,3653,0
7,Sydney,True,3653,3653,0
8,Cairo,True,3653,3653,0
9,Nairobi,True,3653,3653,0


In [14]:
frames = []
for row in cities.itertuples():
    raw_file = RAW / f"{row.city.replace(' ', '_')}.json"
    if not raw_file.exists():
        continue
    data = json.loads(raw_file.read_text())
    df = pd.DataFrame(data["daily"]).rename(columns={"time": "date"})
    df.insert(0, "city", row.city)
    df.insert(1, "country", row.country)
    df.insert(2, "latitude", row.latitude)
    df.insert(3, "longitude", row.longitude)
    frames.append(df)

weather = pd.concat(frames, ignore_index=True)
weather["date"] = pd.to_datetime(weather["date"])
weather.to_csv(PROCESSED / "weather_combined.csv", index=False)

print("Shape:", weather.shape)
print("Cities:", weather["city"].nunique())
print("Date range:", weather["date"].min().date(), "to", weather["date"].max().date())
weather.head()

Shape: (73060, 23)
Cities: 20
Date range: 2016-01-01 to 2025-12-31


,city,country,latitude,longitude,date,weather_code,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,...,wind_gusts_10m_max,wind_direction_10m_dominant,shortwave_radiation_sum,sunshine_duration,daylight_duration,relative_humidity_2m_mean,dew_point_2m_mean,cloud_cover_mean,pressure_msl_mean,surface_pressure_mean
0,Lahore,Pakistan,31.5497,74.3436,2016-01-01,3,20.9,8.9,14.5,0.0,...,14.4,340,10.82,35639.67,36470.50,64,7.2,46,1021.0,994.5
1,Lahore,Pakistan,31.5497,74.3436,2016-01-02,3,20.6,6.2,13.0,0.0,...,19.4,351,12.30,35442.13,36500.41,69,6.8,37,1021.9,995.3
2,Lahore,Pakistan,31.5497,74.3436,2016-01-03,3,21.6,7.5,14.1,0.0,...,21.6,298,12.20,35613.56,36532.97,65,6.9,34,1018.7,992.3
3,Lahore,Pakistan,31.5497,74.3436,2016-01-04,51,20.9,9.5,14.6,0.6,...,16.6,99,9.30,28133.52,36568.10,73,9.5,63,1018.7,992.3
4,Lahore,Pakistan,31.5497,74.3436,2016-01-05,3,21.9,8.8,14.8,0.0,...,21.6,354,12.77,35593.52,36605.75,77,10.3,27,1018.6,992.2


### Data dictionary

| Column | Meaning | Unit | Type |
|---|---|---|---|
| city, country | Location names | - | Categorical |
| latitude, longitude | Coordinates | degrees | Numerical |
| date | Calendar day (local time) | - | Datetime |
| weather_code | WMO weather condition code | code | Categorical (numeric code) |
| temperature_2m_max / min / mean | Daily max/min/mean temperature at 2 m | °C | Numerical |
| precipitation_sum | Total daily precipitation | mm | Numerical |
| rain_sum | Daily rain only (excludes snow) | mm | Numerical |
| precipitation_hours | Hours with precipitation | h | Numerical |
| wind_speed_10m_max | Max wind speed at 10 m | km/h | Numerical |
| wind_gusts_10m_max | Max wind gusts at 10 m | km/h | Numerical |
| wind_direction_10m_dominant | Dominant wind direction | ° (0-360) | Numerical (circular) |
| shortwave_radiation_sum | Total solar radiation | MJ/m² | Numerical |
| sunshine_duration | Seconds of sunshine | s | Numerical |
| daylight_duration | Seconds of daylight | s | Numerical |

**Data source note:** Data comes from the Open-Meteo Historical Weather API, which uses reanalysis datasets. These combine observations with numerical weather models, so values are estimates for a grid location and are not identical to measurements from a weather station. Attribution: Open-Meteo (https://open-meteo.com).